# 007: Weight Initialization Theory — Xavier and He derivations, variance preservation across layers

## 1. THE INITIALIZATION PROBLEM
- If weights are initialized too large, activations explode exponentially through layers (**exploding gradients**).
- If weights are initialized too small, activations shrink toward zero (**vanishing gradients**).
- **Goal**: Initialize weights so that the **variance of activations remains stable** across layers.

## 2. XAVIER INITIALIZATION (Glorot, 2010)
- For layers with **sigmoid/tanh** activations.
- **Derivation**: Assuming inputs have zero mean and unit variance, the variance of the output of a linear layer is:
  $$\text{Var}(z) = n_{\text{in}} \cdot \text{Var}(w) \cdot \text{Var}(x)$$
- To preserve variance ($\text{Var}(z) = \text{Var}(x)$): $\text{Var}(w) = \frac{1}{n_{\text{in}}}$.
- **Xavier Uniform**: $W \sim U\left[-\sqrt{\frac{6}{n_{\text{in}}+n_{\text{out}}}}, \sqrt{\frac{6}{n_{\text{in}}+n_{\text{out}}}}\right]$

## 3. HE INITIALIZATION (He, 2015)
- For layers with **ReLU** activations. ReLU zeroes out ~50% of neurons, halving the effective fan-in.
- **Correction**: $\text{Var}(w) = \frac{2}{n_{\text{in}}}$

---


In [ ]:
import numpy as np

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def relu(z):
    return np.maximum(0, z)

def test_initialization(init_type, activation, layer_dims, n_samples=500):
    """Measures activation variance across layers for a given initialization."""
    np.random.seed(0)
    X = np.random.randn(layer_dims[0], n_samples)  # Input
    A = X
    
    variances = [np.var(A)]
    
    for l in range(1, len(layer_dims)):
        fan_in = layer_dims[l-1]
        fan_out = layer_dims[l]
        
        if init_type == "zeros":
            W = np.zeros((fan_out, fan_in))
        elif init_type == "large":
            W = np.random.randn(fan_out, fan_in) * 1.0
        elif init_type == "xavier":
            W = np.random.randn(fan_out, fan_in) * np.sqrt(1.0 / fan_in)
        elif init_type == "he":
            W = np.random.randn(fan_out, fan_in) * np.sqrt(2.0 / fan_in)
        
        Z = W @ A
        A = activation(Z)
        variances.append(np.var(A))
    
    return variances



In [ ]:
print("--- Activation Variance Across 10 Layers ---")
dims = [256] * 11  # 10 hidden layers, 256 neurons each

for init in ["large", "xavier", "he"]:
    for act_name, act_fn in [("sigmoid", sigmoid), ("relu", relu)]:
        vars_ = test_initialization(init, act_fn, dims)
        print(f"  {init:>7} + {act_name:>7} → Var[layer1]={vars_[1]:.4f}, Var[layer5]={vars_[5]:.6f}, Var[layer10]={vars_[10]:.8f}")

print("\n[Insight] Xavier + sigmoid preserves variance. He + relu preserves variance.")
print("[Insight] Large init + sigmoid → variance explodes then saturates to ~0.25.")
